# Run the standard experiment

Run the prepared input through HP search and eta selection, adaptation and
classifier training, initial/raw/MLP baselines, and saved-result checks.
All execution switches default to **False**.

## Use

Edit Settings, save this notebook, restart the kernel, and run all cells from
the top. New experiments use a separate output directory by default.

To inspect an existing experiment, set `EXPERIMENT_ROOT` and leave all execution
switches False. To resume from a saved HP selection, set `SELECTION_SOURCE` and
enable only the required stages. No stage automatically runs an earlier stage.

Use a new `EXPERIMENT_ROOT` when scientific conditions or saving options change.
Use a new `EXECUTION_NAME` when settings, source code, or execution switches
change. Matching completed conditions are reused; incomplete conditions restart
from the beginning. Return all execution switches to False after computation.

Detailed plots remain in notebooks 01-05. This notebook does not provide
readout-only retraining or encoder-reuse experiments.

In [1]:
from dataclasses import asdict, replace
from pathlib import Path
import hashlib
import importlib
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'experiment_settings.py').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'experiment_settings.py').is_file():
    raise FileNotFoundError('Open this notebook from the project root or notebooks directory')
sys.path[:0] = [str(ROOT), str(ROOT / 'src')]

import pandas as pd
import torch
from IPython.display import display
import experiment_settings as shared
from hnn2.config import RunSpec, experiment_config_from_json
from hnn2.data import load_dataset
from hnn2.hp.select import eta_table, landscape_table
from hnn2.postprocess import compare_tables
from hnn2.result_io import write_result
from hnn2.workflows import (
    baseline_population, read_selection, run_baselines, run_selected_experiments,
    run_sweep_selection, save_expected_runs, _check_training_config,
)

shared = importlib.reload(shared)
NOTEBOOK_PATH = ROOT / 'notebooks/run_experiment.ipynb'

## 1. Settings and local changes

Load the shared defaults and define notebook-specific changes here. Use `replace`
to modify frozen dataclasses; do not edit `experiment_settings.py`.

The HP sweep and training must use the same model and adaptation settings.
Their epoch counts and `readout_*` settings may differ. For example, changing
the classifier budget in `CONFIG` does not change the HP eta-selection rule.

`SELECTION_SOURCE=None` uses this experiment's `results/hp/selection`. To resume
from another saved selection, set its path and keep `RUN_HP=False`. The saved
selection supplies the model, target, and seed population. A baseline-only run
uses the same population when a selection is provided; otherwise, it uses local
`MODELS` and `SEEDS`.

Each enabled execution saves its effective settings and copies of this notebook
and the shared settings under `results/notebook_runs/<EXECUTION_NAME>/`. Use a
new execution name when settings, switches, or notebook code change.

The execution record's `COMPLETE` marker confirms that the record was saved; it
does not confirm that every experiment run finished. Actual scientific settings
and completion are recorded separately for each run.

In [2]:
DATASET = shared.DATASET
EXPERIMENT_ROOT = shared.OUTPUT_ROOT / f'{shared.EXPERIMENT_NAME}_notebook_v1'
SELECTION_SOURCE = None
EXECUTION_NAME = 'run_v1'

MODELS = shared.MODELS
TARGETS = shared.SWEEP_TARGETS
ETAS = shared.SWEEP_ETAS
SEEDS = shared.SEEDS
CONFIG = shared.CONFIG
SWEEP_CONFIG = shared.SWEEP_CONFIG
RULE = shared.RULE
ANALYSIS = shared.ANALYSIS
MLP = shared.MLP
SAVING = shared.SAVING
BASELINE_REPRESENTATIONS = shared.BASELINE_REPRESENTATIONS
DEVICE = shared.DEVICE
BATCH_RUNS = shared.BATCH_RUNS
TORCH_THREADS = shared.TORCH_THREADS

RUN_HP = False
RUN_TRAINING = False
RUN_BASELINES = False

## 2. Prepared data and the intended work

This step reads prepared arrays; it never downloads data. Inspect images and class
balance with `00_mnist_overview.ipynb`. Seed values below are indices of the
project's named random streams, not raw random seeds.

Review the effective settings, destination and condition counts before enabling
computation. Missing results are normal for a new destination. An explicit saved
selection must be complete and match the input and training conditions.

In [3]:
# Build output paths and choose the HP selection to use.
EXPERIMENT_ROOT = Path(EXPERIMENT_ROOT).expanduser().resolve()
RESULTS = EXPERIMENT_ROOT / 'results'
HP_CANDIDATES = RESULTS / 'hp/candidates'
HP_OUTPUT = RESULTS / 'hp/selection'
HP_SOURCE = (
    HP_OUTPUT
    if SELECTION_SOURCE is None
    else Path(SELECTION_SOURCE).expanduser().resolve()
)
TRAINING_OUTPUT = RESULTS / 'plasticity'
BASELINES_OUTPUT = RESULTS / 'baselines'
EXECUTION_PATH = RESULTS / 'notebook_runs' / EXECUTION_NAME

# Keep each execution record directly under notebook_runs.
if (
    not EXECUTION_NAME
    or Path(EXECUTION_NAME).name != EXECUTION_NAME
    or EXECUTION_NAME in ('.', '..')
):
    raise ValueError('Use a single folder name for EXECUTION_NAME')

# Reject ambiguous switches and invalid parallelism settings.
if any(type(flag) is not bool for flag in (RUN_HP, RUN_TRAINING, RUN_BASELINES)):
    raise ValueError('Execution switches must be True or False')

if any(type(n) is not int or n < 1 for n in (BATCH_RUNS, TORCH_THREADS)):
    raise ValueError('BATCH_RUNS and TORCH_THREADS must be positive integers')

# Newly computed HP results must also be the selection used by this experiment.
if RUN_HP and HP_SOURCE != HP_OUTPUT:
    raise ValueError('Set SELECTION_SOURCE=None when computing HP in this experiment')

In [4]:
# Load prepared arrays and their source information.
dataset = load_dataset(DATASET)

# Inspect split sizes and data types before starting computation.
display(pd.DataFrame([
    {'array': name, 'shape': value.shape, 'dtype': str(value.dtype)}
    for name, value in dataset.arrays.items()
]))

print('Input:', dataset.source['path'])
print('Input SHA256:', dataset.source['sha256'])  # Identifies the input file contents.
print('Results:', RESULTS)

,array,shape,dtype
0,x_train_small,"(1024, 100)",float32
1,x_train_big,"(1024, 484)",float32
2,y_train,"(1024,)",int64
3,x_val_small,"(1024, 100)",float32
4,x_val_big,"(1024, 484)",float32
5,y_val,"(1024,)",int64
6,x_test_small,"(1024, 100)",float32
7,x_test_big,"(1024, 484)",float32
8,y_test,"(1024,)",int64
9,x_test_full_small,"(10000, 100)",float32


Input: data\mnist_seed-0.npz
Input SHA256: d00f4244c9437f99ea96303bd3fffaa695162d636b776f980bb7954916ddad1a
Results: outputs\synthetic_long_v1_notebook_v1\results


In [5]:
# Enumerate every requested model, target, learning rate, and seed combination.
SWEEP_SPECS = [
    RunSpec(model, target, eta, seed)
    for model in MODELS
    for target in TARGETS
    for eta in ETAS
    for seed in SEEDS
]

selection, selected_specs = None, None

# A saved selection must match the input and intended training conditions.
if (HP_SOURCE / 'COMPLETE').is_file():
    selection, selected_specs = read_selection(HP_SOURCE)

    if selection['input']['sha256'] != dataset.source['sha256']:
        raise ValueError('Saved HP selection and prepared input differ')

    _check_training_config(
        experiment_config_from_json(json.dumps(selection['sweep_config'])),
        CONFIG,
    )

    if CONFIG.n_epochs != selection['schedule']['n_epochs']:
        raise ValueError('Training epochs differ from the saved HP schedule')

elif SELECTION_SOURCE is not None or (RUN_TRAINING and not RUN_HP):
    raise ValueError('Choose a complete HP selection or explicitly run HP first')

# A new sweep must cover the training budget and the HP stability window.
if RUN_HP:
    _check_training_config(SWEEP_CONFIG, CONFIG)

    if not RULE.at_epoch < CONFIG.n_epochs <= SWEEP_CONFIG.n_epochs:
        raise ValueError('Require HP at_epoch < training epochs <= sweep epochs')

    if RULE.at_epoch + RULE.stability_epochs >= SWEEP_CONFIG.n_epochs:
        raise ValueError('The sweep must cover the full stability window')

In [6]:
# When reusing HP results, inherit their population instead of local candidates.
population_specs = (
    SWEEP_SPECS
    if RUN_HP or selected_specs is None
    else selected_specs
)

# Baselines use the same model and seed sets as the active population.
active_models = list(dict.fromkeys(
    spec.model_code for spec in population_specs
))
active_seeds = sorted({
    spec.seed_index for spec in population_specs
})

baseline_rows = baseline_population(
    BASELINES_OUTPUT,
    active_models,
    active_seeds,
    BASELINE_REPRESENTATIONS,
)

# Count planned conditions; completion and reuse are checked during execution.
display(pd.DataFrame([
    {
        'stage': 'HP candidates',
        'conditions': len(SWEEP_SPECS),
        'execute': RUN_HP,
    },
    {
        'stage': 'Training',
        # Training uses one selected eta per model and target.
        'conditions': len({
            (s.model_code, s.targ, s.seed_index)
            for s in population_specs
        }),
        'execute': RUN_TRAINING,
    },
    {
        'stage': 'Baselines',
        'conditions': len(baseline_rows),
        'execute': RUN_BASELINES,
    },
]))

,stage,conditions,execute
0,HP candidates,1296,False
1,Training,81,False
2,Baselines,15,False


In [7]:
# Compare training and HP settings side by side, including notebook overrides.
display(pd.DataFrame({
    'training': asdict(CONFIG),
    'HP': asdict(SWEEP_CONFIG),
}))

# Review selection, saving, representation learning, and analysis settings.
display(pd.Series(asdict(RULE), name='HP selection rule').to_frame())
display(pd.Series(asdict(SAVING), name='Save numeric outputs').to_frame())
display(pd.Series(asdict(MLP), name='MLP settings').to_frame())
display(pd.Series(asdict(ANALYSIS), name='Analysis settings').to_frame())

,training,HP
schema_version,1,1
image_size,"(10, 10)","(10, 10)"
raw_baseline_size,"(22, 22)","(22, 22)"
n_samples_per_split,1024,1024
train_val_ratio,0.9,0.9
data_seed,0,0
n_excitatory,484,484
n_inhibitory,1,1
tau_e,2.0,2.0
tau_i,20.0,20.0


,HP selection rule
at_epoch,7.0
var_weight,1.0
stability_epochs,2.0
stability_tol,0.5


,Save numeric outputs
save_features,True
save_full_test,True
save_encoder_weights,True
save_activity,True
save_summaries,True
save_readout_history,True


,MLP settings
epochs,15.000
lr,0.003
batch_size,64.000


,Analysis settings
split,test
population_bins,"{'lo': 0.0, 'hi': 6.0, 'width': 0.05}"
lifetime_bins,"{'lo': 0.0, 'hi': 3.0, 'width': 0.05}"
allow_dropped,True
metric,cosine
center,False
metrics,"(entropy, silhouette)"


## 3. Record this execution

With every switch False, this cell writes nothing. When enabled, save the notebook
file first so the copied code includes your edits. The record stores actual
runtime settings, the input identity, any reused HP selection, and source hashes.
Rerunning an identical record leaves it unchanged; changed settings require a new
execution name. The experiment APIs check existing conditions before reuse and
refuse incompatible completed outputs.

In [8]:
effective_settings = {
    # Record the input, output location, and any reused HP selection.
    'dataset': dataset.source,
    'experiment_root': str(EXPERIMENT_ROOT),
    'hp_source': str(HP_SOURCE),
    'selection': selection.get('source') if selection and not RUN_HP else None,

    # Keep local candidates and the active population separately.
    'models': list(MODELS),
    'targets': list(TARGETS),
    'etas': list(ETAS),
    'seeds': list(SEEDS),
    'active_models': active_models,
    'active_seeds': active_seeds,

    # Capture effective settings, including notebook overrides.
    'config': asdict(CONFIG),
    'sweep_config': asdict(SWEEP_CONFIG),
    'rule': asdict(RULE),
    'analysis': asdict(ANALYSIS),
    'mlp': asdict(MLP),
    'saving': asdict(SAVING),
    'baseline_representations': list(BASELINE_REPRESENTATIONS),

    # Record execution controls separately from scientific conditions.
    'device': str(DEVICE),
    'batch_runs': BATCH_RUNS,
    'torch_threads': TORCH_THREADS,
    'run_hp': RUN_HP,
    'run_training': RUN_TRAINING,
    'run_baselines': RUN_BASELINES,

    # Hash the files saved on disk; unsaved notebook edits are not included.
    'notebook_sha256': hashlib.sha256(NOTEBOOK_PATH.read_bytes()).hexdigest(),
    'shared_settings_sha256': hashlib.sha256(
        shared.SETTINGS_PATH.read_bytes()
    ).hexdigest(),
}

# Normalize tuples to JSON lists for comparison with saved records.
# Reject NaN and infinity rather than writing nonstandard JSON.
effective_settings = json.loads(
    json.dumps(effective_settings, allow_nan=False)
)

In [9]:
# Create execution records only when at least one computation stage is enabled.
if RUN_HP or RUN_TRAINING or RUN_BASELINES:

    # Check the requested device before writing the execution record.
    if str(DEVICE).startswith('cuda') and not torch.cuda.is_available():
        raise RuntimeError(
            'CUDA is unavailable; explicitly choose an available device in Settings'
        )
    torch.set_num_threads(TORCH_THREADS)

    # Reuse an execution name only when its complete record matches exactly.
    if EXECUTION_PATH.exists():
        if not (EXECUTION_PATH / 'COMPLETE').is_file():
            raise ValueError(
                'Incomplete execution record; choose a new EXECUTION_NAME'
            )

        previous = json.loads(
            (EXECUTION_PATH / 'effective_settings.json').read_text(
                encoding='utf-8'
            )
        )
        if previous != effective_settings:
            raise ValueError(
                'Execution settings changed; choose a new EXECUTION_NAME'
            )

    else:
        # Save effective settings together with notebook and shared-setting copies.
        # This record's COMPLETE marker does not indicate completed training.
        write_result(
            EXECUTION_PATH,
            {
                'representation': 'notebook_execution',
                'input': dataset.source,
            },
            documents={'effective_settings': effective_settings},
            script_path=NOTEBOOK_PATH,
            settings_path=shared.SETTINGS_PATH,
        )

    # Preserve intended baselines even if HP or training stops before that stage.
    save_expected_runs(BASELINES_OUTPUT, baseline_rows, config=CONFIG)
    print('Execution settings:', EXECUTION_PATH)

else:
    print('Preview only: no execution record or result directory was created.')

Preview only: no execution record or result directory was created.


## 4. HP search and eta selection

`RUN_HP=True` searches the displayed candidates and applies the unchanged eta
selection rule. Matching complete candidates/selection are reused. With the
switch False, only saved selection evidence is read. The full landscape and
curves remain available as DataFrames; only short previews are displayed.

Changing the target display in a figure notebook does not change this HP search.
Detailed HP plots are in `01_selection.ipynb`.

In [10]:
# Run HP search and eta selection, reusing matching completed results.
if RUN_HP:
    run_sweep_selection(
        dataset, SWEEP_SPECS, SWEEP_CONFIG, RULE, HP_CANDIDATES, HP_OUTPUT,
        training_config=CONFIG, batch_runs=BATCH_RUNS, device=DEVICE,
        script_path=NOTEBOOK_PATH, settings_path=shared.SETTINGS_PATH,
    )

# Load and validate the saved selection, including when computation is disabled.
selection, selected_specs = None, None
if (HP_SOURCE / 'COMPLETE').is_file():
    selection, selected_specs = read_selection(HP_SOURCE)

    # Show selected etas and previews of candidate scores and monitor curves.
    selected_eta = eta_table(selection)
    hp_landscape = landscape_table(selection)
    hp_curves = pd.read_csv(
        HP_SOURCE / 'curves.csv', float_precision='round_trip'
    )
    display(selected_eta)
    display(hp_landscape.head(12))
    display(hp_curves.head(12))

    # Align baseline models and seeds with the saved HP population.
    active_models = list(dict.fromkeys(
        spec.model_code for spec in selected_specs
    ))
    active_seeds = sorted({
        spec.seed_index for spec in selected_specs
    })

else:
    print('No completed HP selection yet. Enable RUN_HP to compute it.')

No completed HP selection yet. Enable RUN_HP to compute it.


## 5. Train the selected conditions

Enable only `RUN_TRAINING` to start from a saved selection after restarting the
kernel. Run setup/preview cells first; the HP cell only reads when `RUN_HP=False`.
This stage performs adaptation, feature extraction, linear-classifier training
and saved metrics for every selected model/target and its original HP seeds.
It does not rerun HP. `last_run.csv` reports the most recent attempt; it is not
the planned population. A retry skips complete matching runs and restarts unfinished runs.

In [11]:
# Train with the saved HP selection, reusing matching completed results.
if RUN_TRAINING:
    training_states = run_selected_experiments(
        HP_SOURCE, dataset, CONFIG, TRAINING_OUTPUT, batch_runs=BATCH_RUNS,
        device=DEVICE, analysis=ANALYSIS, saving=SAVING,
        script_path=NOTEBOOK_PATH, settings_path=shared.SETTINGS_PATH,
    )
    display(training_states)

# When training is disabled, show the recorded status without recomputation.
elif (TRAINING_OUTPUT / 'last_run.csv').is_file():
    print('Saved status from the latest training attempt:')
    display(pd.read_csv(
        TRAINING_OUTPUT / 'last_run.csv',
        keep_default_na=False,  # Preserve empty reason fields as empty strings.
    ))

else:
    print('Training is OFF; no saved training attempt was found.')

Training is OFF; no saved training attempt was found.


## 6. Initial, raw and MLP baselines

Enable `RUN_BASELINES` to compute these comparisons without starting HP or
adaptation. Initial sparse representations are per model/seed; raw and MLP
are shared per seed, with no target axis. All include a linear classifier.
`MLP.epochs` trains the MLP representation; `CONFIG.readout_epochs` controls
the subsequent linear classifier. Their budgets are different settings.

In [12]:
# Build all requested baselines, reusing matching completed results.
if RUN_BASELINES:
    baseline_states = run_baselines(
        dataset, active_models, active_seeds, CONFIG, BASELINES_OUTPUT,
        representations=BASELINE_REPRESENTATIONS, mlp=MLP,
        analysis=ANALYSIS, device=DEVICE, saving=SAVING,
        script_path=NOTEBOOK_PATH,
        settings_path=shared.SETTINGS_PATH,
    )
    display(baseline_states)

# When baseline computation is disabled, show the latest recorded status.
elif (BASELINES_OUTPUT / 'last_run.csv').is_file():
    print('Saved status from the latest baseline attempt:')
    display(pd.read_csv(
        BASELINES_OUTPUT / 'last_run.csv',
        keep_default_na=False,  # Preserve empty reason fields as empty strings.
    ))

else:
    print('Baselines are OFF; no saved baseline attempt was found.')

Baselines are OFF; no saved baseline attempt was found.


## 7. Read saved results and inspect coverage

Reload saved summaries rather than relying on in-memory training return values.
The status table below reads the recorded expected population and completion
markers. Missing/failed conditions stay visible. It is a status overview, not a
full array-validity check. Figure notebooks perform their required input checks.
Per-run rows retain seed, target, eta and classifier-budget columns; no targets
or seeds are silently pooled. Saved test accuracy does not select a target.

In [13]:
# Compare each planned run with the markers currently saved on disk.
status_rows = []

for directory in (TRAINING_OUTPUT, BASELINES_OUTPUT):
    expected_path = directory / 'expected_runs.csv'

    # A stage without an expected-run table has not registered its population.
    if not expected_path.is_file():
        continue

    planned = pd.read_csv(
        expected_path,
        float_precision='round_trip',
    )

    for row in planned.to_dict('records'):
        run_path = Path(row['path'])
        state, reason = 'missing', ''

        # COMPLETE is the authoritative marker for a finished run.
        if (run_path / 'COMPLETE').is_file():
            state = 'complete'

        # Preserve the recorded failure message for inspection.
        elif (run_path / 'FAILED.json').is_file():
            state = 'failed'
            failure = json.loads(
                (run_path / 'FAILED.json').read_text(encoding='utf-8')
            )
            reason = failure.get('message', '')

        # An existing folder without COMPLETE or FAILED is unfinished.
        elif run_path.exists():
            state = 'incomplete'

        status_rows.append({
            **row,
            'state': state,
            'reason': reason,
        })

status = pd.DataFrame(status_rows)

In [14]:
# Start with an empty table so later cells can use summary consistently.
summary = pd.DataFrame()

if not status.empty:
    # Show the number of runs in each representation and state.
    status_counts = (
        status
        .groupby(['representation', 'state'], dropna=False)
        .size()
        .rename('runs')
        .to_frame()
    )
    display(status_counts)

    # Make missing, incomplete, and failed runs visible.
    display(status.loc[status.state != 'complete'])

    # Read numerical summaries only from runs marked complete.
    complete_paths = status.loc[
        status.state == 'complete',
        'path',
    ].tolist()

    if complete_paths:
        summary = compare_tables(complete_paths, 'summary')

        # Keep the preview compact while retaining the source directory.
        columns = [
            'representation',
            'model_code',
            'targ',
            'eta',
            'seed_index',
            'final_val_loss',
            'final_val_accuracy',
            'final_test_accuracy',
            'config.readout_epochs',
            'source_directory',
        ]
        display(
            summary[
                [name for name in columns if name in summary.columns]
            ].head(30)
        )

else:
    print(
        'No expected-run tables yet. '
        'No results were inferred from folder names.'
    )

No expected-run tables yet. No results were inferred from folder names.


## 8. Continue with the saved-figure notebooks

Open the notebook for your question and set its `RESULTS` to the path printed
below. When reusing an external HP selection, change the first
`selection_directory = ...` assignment in the figure notebook's input-loading
cell to the printed `HP_SOURCE` path. Also choose a separate `OUTPUT`. These notebooks need the
complete required population and saved arrays; a partial experiment may not
meet all 34 figure IDs. Inspect its status and summaries above first.

| Notebook | Purpose |
|---|---|
| 01_selection | HP selection, target evidence and target response |
| 02_adaptation | Monitors, parameter trajectories and optional extra inference |
| 03_distributions | Activity distributions, S scores and bin diagnostics |
| 04_classifier | Final accuracy, paired differences and measured curves |
| 05_geometry | Silhouette, endpoint checks and optional UMAP |

Those notebooks remain read-only by default, with PDF OFF. Their optional
inference/UMAP steps have separate switches and fresh output destinations.
For this notebook, return all three execution switches to False and save the
file when finished. Restart from Settings whenever inputs or conditions change.

In [15]:
print('Figure RESULTS:', RESULTS)
print('Figure selection_directory:', HP_SOURCE)
print('Suggested figure OUTPUT:', EXPERIMENT_ROOT / 'analysis/notebook_figures_v2')

Figure RESULTS: outputs\synthetic_long_v1_notebook_v1\results
Figure selection_directory: outputs\synthetic_long_v1_notebook_v1\results\hp\selection
Suggested figure OUTPUT: outputs\synthetic_long_v1_notebook_v1\analysis\notebook_figures_v2
